# Notebook 2 of 3 — What Happens When the Model's World Changes?

## The Scenario

Imagine it's June 2022. You've spent months building a model that forecasts dwelling approvals per LGA. You've trained it on three years of historical data. You've tested it. It performs well. You deploy it.

Then, in August 2022, two things happen almost simultaneously:
1. The Australian government announces the National Housing Accord — a major housing policy shift
2. Post-COVID construction costs, already rising, peak at ~15–20% YoY growth — the highest in decades

Your model was never exposed to either of these conditions. It learned what "normal" looks like. Now normal is gone.

**This notebook asks and answers three questions:**
1. What does the model's error look like in the months after the shift?
2. How quickly can the monitoring system detect that something is wrong — *without waiting for actual approval data to come in*?
3. How much earlier does the early warning arrive compared to the confirmation signal?

---

## Why This Matters for Housing Supply Forecasting

A housing supply forecast that silently drifts is operationally damaging. Infrastructure planners, developers, and government agencies make multi-year investment decisions based on approval forecasts. A model that was accurate before 2022 but degrades silently afterward — while still producing confident-looking numbers — could lead to:

- Overestimation of supply in councils facing construction cost headwinds
- Underestimation of the gap between Accord targets and actual delivery
- Delayed retraining decisions that compound the error

The monitoring system this notebook demonstrates is designed to catch that drift in real time — before it shows up in reports.

---

## What You Will See

| Section | What it demonstrates |
|---|---|
| **1. Load data** | Establish the pre/post-2022 world in the data |
| **2. Train model (pre-2022)** | Freeze training at the last "safe" quarter |
| **3. Post-2022 predictions** | See the miss against actuals |
| **4. Feature drift** | Early warning fires — no ground truth needed |
| **5. Residual drift** | Confirmation fires — once actuals arrive |
| **Business impact** | What this means for a planning team |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler

from models.lstm import HousingLSTM, _make_sequences, fit as lstm_fit
from monitoring.drift import detect_feature_drift, _FEATURE_DRIFT_Z_THRESHOLD

FEATURES_PATH = Path('../data/processed/features.parquet')
TRAIN_END = '2022Q2'

## 1. Setting the Scene: Two Worlds in the Data

We need to be precise about what "pre-2022" and "post-2022" mean in this data.

The flag `post_accord_2022` is 0 for all financial years up to and including FY2021–22 (i.e., up to 2022Q2), and 1 from FY2022–23 onward. **The model sees only the 0s during training. The 1s are the test environment it was never exposed to.**

The table below shows what national approvals and construction cost growth looked like right around the transition. Notice the `construction_cost_yoy` values — that's the variable that will drive the drift detection later.

In [ ]:
df = pd.read_parquet(FEATURES_PATH)
df['quarter'] = pd.PeriodIndex(df['quarter'], freq='Q')
print(f"Data covers {df['quarter'].min()} to {df['quarter'].max()}")

# Aggregate to national level for visualisation
national = df.groupby('quarter').agg(
    approvals=('dwellings_approved', 'sum'),
    construction_cost_yoy=('construction_cost_yoy', 'first'),
    post_accord_2022=('post_accord_2022', 'first'),
).reset_index()
national['quarter_dt'] = national['quarter'].dt.to_timestamp()
national.tail(8)

## 2. Training the Model on the "Old World"

We deliberately stop training at **Q2 2022**. This is not arbitrary — it replicates the situation of any real deployed model: trained on historical data, deployed before the structural shift occurs.

> *Think of it like a doctor trained on patient data from before a new disease emerged. The doctor's knowledge is real and valid — but it was learned in a world that no longer fully applies.*

The model will learn patterns from:
- **`approvals_lag1`** — how much this LGA approved last year (planning momentum)
- **`population_growth_yoy`** — how fast this LGA's population is growing (demand pressure)
- **`construction_cost_yoy`** — how much more expensive building has become (supply constraint)
- **Seasonal dummies** — which quarter of the year this observation falls in

What it will NOT learn: that construction costs are about to spike to 15–20%, or that a national housing policy commitment is incoming. It will meet those facts for the first time in production.

## 2. Train LSTM through Q2 2022 (pre-break period)

## 3. The Forecast Miss: What the Model Got Wrong

The chart below shows **predicted vs actual approvals** in the post-2022 period. Each point represents the model's output against what ABS data shows actually happened.

In a well-calibrated, stable model:
- The blue line (actuals) and orange line (predictions) should track each other closely
- Errors should be random — sometimes over, sometimes under, no systematic bias

If you see the orange line consistently above or below the blue line — or diverging over time — that is evidence of concept drift: the model's learned relationships no longer match the real world.

In [ ]:
FEATURE_COLS = [
    'approvals_lag1',
    'population_growth_yoy',
    'construction_cost_yoy',
    'season_q1', 'season_q2', 'season_q3', 'season_q4',
    'post_accord_2022',
]
available = [c for c in FEATURE_COLS if c in df.columns]

train_mask = df['quarter'] <= TRAIN_END
val_mask = df['quarter'] > TRAIN_END

X_train_raw = df[train_mask][available].values.astype(np.float32)
y_train_raw = df[train_mask]['dwellings_approved'].values.astype(np.float32)
X_val_raw = df[val_mask][available].values.astype(np.float32)
y_val_raw = df[val_mask]['dwellings_approved'].values.astype(np.float32)

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()
X_train = x_scaler.fit_transform(X_train_raw)
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
X_val = x_scaler.transform(X_val_raw)
y_val = y_scaler.transform(y_val_raw.reshape(-1, 1)).ravel()

model = HousingLSTM(input_size=len(available), hidden_size=32, num_layers=1, output_steps=4)
history = lstm_fit(model, X_train, y_train, X_val, y_val,
                   seq_len=8, horizon=4, batch_size=16, max_epochs=30, patience=5)

print(f"Training complete. Best val MAE: {history['best_val_mae']:.4f}")

## 4. The Early Warning: Feature Drift (No Ground Truth Required)

This is the key innovation in the monitoring system.

**Feature drift in plain English:** The model was trained when construction costs grew at roughly 1–5% per year. When that number jumped to 15–20%, we're asking the model to operate in conditions it has never seen. The model can still compute a number — but its assumptions about how construction costs relate to approvals were learned in a different world.

The monitoring system measures this by computing a **z-score**: how many standard deviations away from the training distribution is the current construction cost growth rate? When the z-score crosses ±2.5, a flag fires.

**What makes this powerful: it requires no ground truth.** You don't need to wait for actual approval data to come in and prove the model is wrong. The moment the input environment shifts — in the same quarter it happens — the flag can fire. This is the early warning, not the post-mortem.

The chart below shows the construction cost z-score over time. Watch for when it crosses the red dashed threshold — that's the monitoring system triggering.

## 3. Show predictions Q3 2022 onwards vs actuals

## 5. The Confirmation: Residual Drift (When the Errors Are Proven)

Feature drift is the suspicion. Residual drift is the proof.

**Residual drift in plain English:** Once actual approval data is available for the post-2022 quarters (typically with a 1–2 quarter lag), we can directly measure whether the model's errors have grown. If the rolling average error (MAE) climbs above 1.5× the training baseline, we trigger a second, harder alert.

This signal is slower — you have to wait for actuals. But it is definitive. It doesn't say "the environment looks different." It says: "We have measured the errors. The model is performing materially worse than it did in testing."

The chart below simulates this rolling MAE signal using the post-2022 prediction errors we computed above.

In [ ]:
model.eval()
device = next(model.parameters()).device

X_vl_seq, y_vl_seq = _make_sequences(X_val, y_val, seq_len=8, horizon=4)
with torch.no_grad():
    preds_scaled = model(X_vl_seq.to(device)).cpu().numpy()

preds = y_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).reshape(preds_scaled.shape)
actuals = y_scaler.inverse_transform(y_vl_seq.numpy().reshape(-1, 1)).reshape(y_vl_seq.shape)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(actuals.mean(axis=1), label='Actual (mean across LGAs)', color='steelblue', marker='o')
ax.plot(preds.mean(axis=1), label='Predicted', color='darkorange', linestyle='--', marker='x')
ax.set_title('Model Predictions vs Actuals: Post-Accord Period (Q3 2022 onwards)')
ax.set_xlabel('Sequence index (quarters after Q3 2022)')
ax.set_ylabel('Dwellings approved (scaled mean)')
ax.legend()
plt.tight_layout()
plt.show()

overall_mae = np.mean(np.abs(actuals - preds))
print(f"Post-Accord MAE: {overall_mae:.4f} (compare to training MAE)")

### The two-signal system: early warning → confirmation

| Signal | When it fires | Ground truth needed | What it tells you |
|---|---|---|---|
| **Feature drift** (z-score) | Same quarter as the shock | ❌ None | "The model's environment has shifted" |
| **Residual drift** (rolling MAE) | 1–2 quarters later | ✅ Actual approvals | "The model's errors have measurably grown" |

The lag between the two signals is operationally valuable:
- **Feature drift** triggers the investigation and the caution flag on outgoing forecasts
- **Residual drift** triggers the retraining decision and the formal model update

In a housing supply context, this means a planning team has **weeks to months** of warning before the errors compound and reach decision-makers downstream. That window is the entire value proposition of a monitoring layer.

---

**The bottom line:** A model without monitoring is a model you trust blindly. A model with this two-tier system is a model you can trust — or know exactly when not to.

## 4. Feature drift flag: construction cost z-score over time

In [ ]:
train_costs = df[df['post_accord_2022'] == 0]['construction_cost_yoy'].dropna()
mean_cost = train_costs.mean()
std_cost = train_costs.std()

macro_sorted = national.sort_values('quarter')
macro_sorted['z_score'] = (macro_sorted['construction_cost_yoy'] - mean_cost) / (std_cost if std_cost > 0 else 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(macro_sorted['quarter_dt'], macro_sorted['z_score'],
        color='steelblue', label='Construction cost z-score')
ax.axhline(_FEATURE_DRIFT_Z_THRESHOLD, color='red', linestyle='--',
           label=f'Drift threshold (z={_FEATURE_DRIFT_Z_THRESHOLD})')
ax.axhline(-_FEATURE_DRIFT_Z_THRESHOLD, color='red', linestyle='--')
ax.axhline(0, color='green', linestyle=':', alpha=0.5, label='Training mean')

breach_mask = macro_sorted['z_score'].abs() > _FEATURE_DRIFT_Z_THRESHOLD
first_breach = macro_sorted[breach_mask]['quarter_dt'].min()
if pd.notna(first_breach):
    label = f'Feature drift triggered: {first_breach.year} Q{first_breach.quarter}'
    ax.axvline(first_breach, color='darkorange', linewidth=2, linestyle=':', label=label)

ax.set_title('Feature Drift: Construction Cost Z-Score vs Pre-Accord Training Distribution')
ax.set_ylabel('Z-score')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Residual drift: rolling MAE over time

In [ ]:
# Compute per-sequence MAE to simulate rolling residual monitoring
seq_mae = np.abs(actuals - preds).mean(axis=1)
training_mae = history['best_val_mae'] * (y_train_raw.max() - y_train_raw.min())  # approximate unscaled
drift_threshold = 1.5  # 1.5x training MAE

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(seq_mae)), seq_mae, color='steelblue', label='Per-quarter MAE')

# Rolling mean
rolling = pd.Series(seq_mae).rolling(4).mean()
ax.plot(range(len(rolling)), rolling, color='steelblue', linestyle='--', linewidth=2, label='Rolling 4-quarter MAE')

baseline = seq_mae[:4].mean() if len(seq_mae) >= 4 else seq_mae.mean()
ax.axhline(baseline, color='green', linestyle=':', label=f'Baseline MAE (first 4 seqs): {baseline:.4f}')
ax.axhline(baseline * drift_threshold, color='red', linestyle='--', label=f'Alert threshold (1.5x): {baseline * drift_threshold:.4f}')

ax.set_title('Residual Drift: Rolling MAE in Post-Rate-Hike Period')
ax.set_xlabel('Quarter index (after Q3 2022)')
ax.set_ylabel('MAE (scaled)')
ax.legend()
plt.tight_layout()
plt.show()

## Business Impact

The 2022 policy-shift is a textbook concept drift case with two compounding factors:

- **Feature drift** (input distribution shift) triggers when house construction cost growth moves outside the training distribution. Post-COVID supply chain stress pushed house construction PPI to ~15–20% YoY growth — well above the pre-2022 training baseline of ~1–5%. The feature drift flag can fire in the same quarter, without requiring ground truth.
- **Residual drift** (degraded prediction quality) follows 1–2 quarters later, once actual approval data confirms the forecast miss.

The production value is the **lag between signals**: feature drift flags the problem before you have ground truth. In a housing market context, a false sense of stability lasts only until the first actuals arrive — roughly 1 quarter behind. A monitoring layer would have alerted a retraining pipeline in the quarter the cost shock occurred, rather than discovering the problem months later when the miss became visible in reports.

For an infrastructure planning team tracking dwelling supply against the National Housing Accord target (1.2 million homes in 5 years), a model that silently degrades over a 6-month period is operationally damaging. Early warning enables:
1. Retraining on updated data before forecast quality visibly drops
2. Flagging model uncertainty to downstream users while retraining is underway
3. Informing decision-makers that the structural environment has changed, not just that forecasts are temporarily unreliable